# 7 - Data transfer and type fidelity

One frame, moved everywhere, with every compromise reported.

In [ ]:
%load_ext econenv
import econenv, numpy as np, pandas as pd, warnings

## A frame with one column of every supported type

In [ ]:
typed = pd.DataFrame({
    'num':   [1.5, 2.5, np.nan, 4.0],
    'whole': pd.array([1, 2, 3, 4], dtype='int64'),
    'txt':   ['a', 'b', '', None],
    'grp':   pd.Categorical(['lo','hi','lo','hi'], categories=['lo','hi'], ordered=True),
    'when':  pd.to_datetime(['2020-01-01','2020-04-01','2020-07-01','2020-10-01']),
    'flag':  [True, False, True, False],
})
typed.dtypes

## Python -> R -> Python is lossless

In [ ]:
econenv.push('r', 'typed', typed)
back = econenv.pull('r', 'typed')
back

In [ ]:
back.dtypes

Including the distinction people usually lose - an **empty string** is still
different from a **missing value**:

In [ ]:
print('row 2 txt:', repr(back['txt'].iloc[2]))
print('row 3 txt:', repr(back['txt'].iloc[3]))
print('categories:', list(back['grp'].cat.categories), '| ordered:', back['grp'].cat.ordered)

## Stata tells you what it cannot keep

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    econenv.push('stata', 'default', typed)
for w in caught:
    print('-', w.message)

## Engine to engine, with no file in between

In [ ]:
moved = econenv.move('stata', 'r', 'default')
econenv.transfer.metadata(moved).history

## What a frame carries with it

In [ ]:
for line in econenv.transfer.dataset_report(back):
    print(line)